In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyproj import Transformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os
import sys

def process_city_data(city_name, input_filename=None, year=None):
    """
    Process building data for a specific city with optional year tagging.
    
    Args:
        city_name (str): Name of the city to process
        input_filename (str, optional): Custom input filename
        year (int, optional): Year to tag this dataset with
    """
    print(f"Processing data for {city_name}...")
    
    # Load data with flexible filename
    if input_filename is None:
        input_file = f"{city_name.lower()}.csv"
    else:
        input_file = input_filename
        
    if not os.path.exists(input_file):
        print(f"Error: Input file {input_file} not found!")
        return
    
    df = pd.read_csv(input_file)
    
    # Convert coordinates
    def convert_coordinates(df):
        transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)
        lon, lat = transformer.transform(df["coordonnee_cartographique_x_ban"].values,
                                         df["coordonnee_cartographique_y_ban"].values)
        df["longitude"] = lon
        df["latitude"] = lat
        return df

    try:
        df = convert_coordinates(df)
    except KeyError as e:
        print(f"Warning: Coordinate conversion failed. Missing column: {e}")
        print("Continuing without coordinate conversion...")
    
    # Rename and engineer features
    df = df.rename(columns={
        "numero_dpe": "building_id",
        "conso_5 usages_ef": "Energy_Consumption",
        "emission_ges_5_usages": "CO2_Usage",
        "etiquette_dpe": "true_energy_label",
        "etiquette_ges": "true_ges_label"
    })
    
    # Check if required columns exist before calculations
    if "Energy_Consumption" in df.columns:
        df["Water_Usage"] = df["Energy_Consumption"] * 0.3
    else:
        print("Warning: Energy_Consumption column not found, skipping Water_Usage calculation")
        
    if "conso_5 usages_par_m2_ef" in df.columns:
        df["Energy_Intensity"] = df["conso_5 usages_par_m2_ef"]
    else:
        print("Warning: conso_5 usages_par_m2_ef column not found")
        
    if "emission_ges_5_usages par_m2" in df.columns:
        df["CO2_Intensity"] = df["emission_ges_5_usages par_m2"]
    else:
        print("Warning: emission_ges_5_usages par_m2 column not found")

    final_cols = [
        #–– Primary key & location ––
        "building_id",
        "latitude", "longitude",

        #–– Core performance metrics ––
        "Energy_Consumption", "CO2_Usage", "Water_Usage",
        "Energy_Intensity", "CO2_Intensity",

        #–– Official DPE labels ––
        "true_energy_label", "true_ges_label",

        #–– BAN address fields (normalized) ––
        "adresse_ban",         # full standardized address
        "numero_voie_ban",     # street number
        "nom_rue_ban",         # street name
        "code_postal_ban",     # postal code
        "nom_commune_ban",     # commune name
        "identifiant_ban",     # BAN address ID

        #–– Raw-fallback address fields ––
        "adresse_brut",
        "nom_commune_brut",
        "code_postal_brut",

        #–– Basic building attributes for filters ––
        "annee_construction",
        "surface_habitable_immeuble",
    ]

    # Filter only columns that exist in the dataframe
    available_cols = [col for col in final_cols if col in df.columns]
    print(f"Available columns: {len(available_cols)}/{len(final_cols)}")
    
    required = [
        "building_id", "Energy_Consumption", "CO2_Usage", "Water_Usage", 
        "Energy_Intensity", "CO2_Intensity"
    ]
    
    # Adjust required columns if location data is missing
    if "latitude" in df.columns and "longitude" in df.columns:
        required.extend(["latitude", "longitude"])
    
    # Check which required columns are available
    available_required = [col for col in required if col in df.columns]
    missing_required = [col for col in required if col not in df.columns]
    
    if missing_required:
        print(f"Warning: Missing required columns: {missing_required}")
    
    # select & only drop rows missing available required columns
    df_clean = df[available_cols].dropna(subset=available_required)
    
    # Add log-transformed features
    numeric_cols = [
        "Energy_Consumption", "CO2_Usage", 
        "Energy_Intensity", "CO2_Intensity"
    ]
    
    # Only transform columns that exist
    existing_numeric = [col for col in numeric_cols if col in df_clean.columns]
    for col in existing_numeric:
        df_clean[f"log1p_{col}"] = np.log1p(df_clean[col])
    
    # Add year tag if provided
    df_clean['city'] = city_name
    if year is not None:
        df_clean['year'] = year
        output_file = f"reduced_{city_name.lower()}_buildings_{year}.csv"
    else:
        output_file = f"reduced_{city_name.lower()}_buildings.csv"
    
    # Save cleaned dataset
    df_clean.to_csv(output_file, index=False)
    print(f"Clean dataset saved to {output_file}.")
    
    return df_clean

def draw_multipliers(n_buildings):
    """
    Generate multipliers for synthetic future data.
    
    Args:
        n_buildings (int): Number of buildings
    
    Returns:
        numpy.ndarray: Array of multipliers
    """
    # choose a distribution "scenario" for each building:
    # 60% get Normal noise, 30% Uniform, 10% Log-Normal "jumps"
    scenarios = np.random.choice(
        ["normal", "uniform", "lognormal"],
        size=n_buildings,
        p=[0.6, 0.3, 0.1]
    )
    m = np.zeros(n_buildings)
    for i, scenario in enumerate(scenarios):
        if scenario == "normal":
            m[i] = np.random.normal(loc=1.0, scale=0.05)
        elif scenario == "uniform":
            m[i] = np.random.uniform(0.9, 1.1)
        else:  # lognormal: small chance of a bigger jump
            m[i] = np.random.lognormal(mean=0, sigma=0.1)
    return m

def generate_future_data(df, city_name, target_year):
    """
    Generate synthetic data for future year based on existing data.
    
    Args:
        df (DataFrame): Source dataframe
        city_name (str): Name of the city
        target_year (int): Target year for synthetic data
    """
    print(f"Generating synthetic data for {city_name}, year {target_year}...")
    
    # Make a copy of the dataframe
    future_df = df.copy()
    future_df['city'] = city_name
    # Update year
    future_df['year'] = target_year
    
    # Get the number of buildings
    n_buildings = len(future_df)
    
    # Generate multipliers for different metrics
    energy_multipliers = draw_multipliers(n_buildings)
    co2_multipliers = draw_multipliers(n_buildings)
    water_multipliers = draw_multipliers(n_buildings)
    
    # Apply multipliers to core metrics
    if "Energy_Consumption" in future_df.columns:
        future_df["Energy_Consumption"] = future_df["Energy_Consumption"] * energy_multipliers
        
    if "Energy_Intensity" in future_df.columns:
        future_df["Energy_Intensity"] = future_df["Energy_Intensity"] * energy_multipliers
        
    if "CO2_Usage" in future_df.columns:
        future_df["CO2_Usage"] = future_df["CO2_Usage"] * co2_multipliers
        
    if "CO2_Intensity" in future_df.columns:
        future_df["CO2_Intensity"] = future_df["CO2_Intensity"] * co2_multipliers
        
    if "Water_Usage" in future_df.columns:
        future_df["Water_Usage"] = future_df["Water_Usage"] * water_multipliers
    
    # Re-calculate log transformations with new values
    numeric_cols = [
        "Energy_Consumption", "CO2_Usage", 
        "Energy_Intensity", "CO2_Intensity"
    ]
    
    # Only transform columns that exist
    existing_numeric = [col for col in numeric_cols if col in future_df.columns]
    for col in existing_numeric:
        log_col = f"log1p_{col}"
        if log_col in future_df.columns:  # Update existing log columns
            future_df[log_col] = np.log1p(future_df[col])
    
    # Generate some random changes in labels if they exist
    if "true_energy_label" in future_df.columns:
        # Randomly improve some labels (5% chance)
        mask = np.random.random(size=n_buildings) < 0.05
        label_map = {"G": "F", "F": "E", "E": "D", "D": "C", "C": "B", "B": "A", "A": "A"}
        future_df.loc[mask, "true_energy_label"] = future_df.loc[mask, "true_energy_label"].map(
            lambda x: label_map.get(x, x) if isinstance(x, str) else x
        )
    
    if "true_ges_label" in future_df.columns:
        # Randomly improve some labels (5% chance)
        mask = np.random.random(size=n_buildings) < 0.05
        future_df.loc[mask, "true_ges_label"] = future_df.loc[mask, "true_ges_label"].map(
            lambda x: label_map.get(x, x) if isinstance(x, str) else x
        )
    
    # Save future dataset
    output_file = f"reduced_{city_name.lower()}_buildings_{target_year}.csv"
    future_df.to_csv(output_file, index=False)
    print(f"Synthetic {target_year} data saved to {output_file}.")
    
    # Run K-Means clustering on the future data
    try:
        cluster_future_data(future_df, city_name, target_year)
    except Exception as e:
        print(f"Warning: Could not perform clustering on future data: {e}")
    
    return future_df

def cluster_future_data(df, city_name, year):
    """
    Apply K-means clustering to future data.
    
    Args:
        df (DataFrame): Data to cluster
        city_name (str): City name
        year (int): Year of data
    """
    # Find log columns for clustering
    log_cols = [col for col in df.columns if col.startswith('log1p_')]
    
    if len(log_cols) < 2:
        print("Not enough log-transformed columns for clustering")
        return df
    
    # Prepare data for clustering
    X = df[log_cols].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # Apply PCA
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(X_scaled)
    df["PC1"], df["PC2"] = pcs[:,0], pcs[:,1]
    
    # Apply K-Means
    kmeans = KMeans(n_clusters=4, random_state=42)
    df["cluster"] = kmeans.fit_predict(X_scaled)
    
    
    
    return df

def analyze_city_data(df, city_name, year=None):
    """
    Analyze the processed city data and generate visualizations.
    
    Args:
        df (DataFrame): Processed dataframe
        city_name (str): City name for titles
        year (int, optional): Year of the data for titles
    """
    title_prefix = f"{city_name}"
    if year is not None:
        title_prefix = f"{city_name} ({year})"
    
    print(f"\nAnalyzing data for {title_prefix}...")
    
    # Quick overview
    print("===== DATA INFO =====")
    df.info()
    print("\n===== MISSING VALUES =====")
    print(df.isnull().sum())
    print("\n===== DESCRIPTIVE STATISTICS =====")
    print(df.describe().T)
    
    # Check for numeric columns availability
    base_numeric_cols = [
        "Energy_Consumption", "CO2_Usage", "Water_Usage",
        "Energy_Intensity", "CO2_Intensity"
    ]
    numeric_cols = [col for col in base_numeric_cols if col in df.columns]
    
    if len(numeric_cols) < 2:
        print("Insufficient numeric columns for analysis.")
        return
    
    # Univariate analysis
    for col in numeric_cols:
        plt.figure(figsize=(8,4))
        sns.histplot(df[col], kde=True)
        plt.title(f"{title_prefix} - {col} — Distribution + KDE")
        plt.tight_layout()
        plt.close()

        plt.figure(figsize=(6,2))
        sns.boxplot(x=df[col])
        plt.title(f"{title_prefix} - {col} — Boxplot")
        plt.tight_layout()
        plt.close()

    # Correlation matrix
    corr = df[numeric_cols].corr()
    print("\n===== CORRELATION MATRIX =====")
    print(corr)
    plt.figure(figsize=(6,5))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
    plt.title(f"{title_prefix} - Correlation Matrix")
    plt.tight_layout()
    plt.close()

    # Pairwise scatterplots
    sns.pairplot(df[numeric_cols], diag_kind="kde", corner=True)
    plt.suptitle(f"{title_prefix} - Pairplot of Numeric Features", y=1.02)
    plt.tight_layout()
    plt.close()

    # Geographic scatter if coordinates are available
    if "longitude" in df.columns and "latitude" in df.columns:
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df["longitude"], y=df["latitude"], 
                        hue=df["Energy_Consumption"] if "Energy_Consumption" in df.columns else None, 
                        palette="viridis", s=20)
        plt.title(f"{title_prefix} - Map Scatter: Energy Consumption")
        plt.tight_layout()
        plt.close()

        # Geo‑bubble plots
        for col in numeric_cols:
            plt.figure(figsize=(6,6))
            sns.scatterplot(x=df["longitude"], y=df["latitude"], 
                            size=df[col], sizes=(10,200), alpha=0.6, legend=False)
            plt.title(f"{title_prefix} - Geo‑Bubble: {col}")
            plt.tight_layout()
            plt.close()

    # Check for existing log columns
    log_cols = [col for col in df.columns if col.startswith('log1p_')]
    
    if log_cols:
        corr2 = df[log_cols].corr()
        plt.figure(figsize=(5,4))
        sns.heatmap(corr2, annot=True, fmt=".2f", cmap="coolwarm", square=True)
        plt.title(f"{title_prefix} - Corr of Log‑Transformed Features")
        plt.close()

        sns.pairplot(df[log_cols], diag_kind="kde", corner=True)
        plt.suptitle(f"{title_prefix} - Pairplot of Log‑Transformed Metrics", y=1.02)
        plt.tight_layout()
        plt.close()
    
    # PCA visualization if PC columns exist
    if "PC1" in df.columns and "PC2" in df.columns:
        plt.figure(figsize=(6,5))
        sns.scatterplot(x="PC1", y="PC2", data=df, alpha=0.3)
        plt.title(f"{title_prefix} - PCA of Building Metrics")
        plt.tight_layout()
        plt.close()
        
        # K-Means visualization if cluster column exists
        if "cluster" in df.columns:
            plt.figure(figsize=(6,5))
            sns.scatterplot(x="PC1", y="PC2", hue="cluster", data=df, palette="tab10", alpha=0.5)
            plt.title(f"{title_prefix} - K‑Means Clusters on PCA space")
            plt.tight_layout()
            plt.close()
    
    return df

def process_city_with_years(city_name, input_filename=None, base_year=2024, future_years=[2025]):
    """
    Process city data and generate data for multiple years.
    
    Args:
        city_name (str): Name of the city
        input_filename (str, optional): Input filename
        base_year (int): Base year for initial data
        future_years (list): List of future years to generate
    
    Returns:
        dict: Dictionary of dataframes by year
    """
    # Process base year data
    base_df = process_city_data(city_name, input_filename, base_year)
    if base_df is None:
        return None
        
    # Apply clustering to base data
    try:
        cluster_future_data(base_df, city_name, base_year)
    except Exception as e:
        print(f"Warning: Could not perform clustering on base data: {e}")
    
    # Store all dataframes in a dictionary
    all_dfs = {base_year: base_df}
    
    # Generate data for future years
    current_df = base_df
    for year in future_years:
        future_df = generate_future_data(current_df, city_name, year)
        all_dfs[year] = future_df
        # Use the latest year as base for the next year (cumulative changes)
        current_df = future_df
    
    # Create a combined dataset with all years
    combined_df = pd.concat(all_dfs.values())
    combined_output = f"reduced_{city_name.lower()}_buildings_all_years.csv"
    combined_df.to_csv(combined_output, index=False)
    print(f"Combined multi-year dataset saved to {combined_output}.")
    
    return all_dfs

In [5]:
city_dfs = process_city_with_years(
    "gordes", 
    input_filename="gordes.csv",  
    base_year=2024, 
    future_years=[2025]
)


Processing data for gordes...
Available columns: 21/21
Clean dataset saved to reduced_gordes_buildings_2024.csv.
Generating synthetic data for gordes, year 2025...
Synthetic 2025 data saved to reduced_gordes_buildings_2025.csv.
Combined multi-year dataset saved to reduced_gordes_buildings_all_years.csv.


c:\Users\USER\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\USER\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


In [7]:
city_dfs = process_city_with_years(
    "lyon", 
    input_filename="Lyon.csv",  
    base_year=2024, 
    future_years=[2025]
)


Processing data for lyon...
Continuing without coordinate conversion...
Available columns: 6/21
Clean dataset saved to reduced_lyon_buildings_2024.csv.
Generating synthetic data for lyon, year 2025...
Synthetic 2025 data saved to reduced_lyon_buildings_2025.csv.
Combined multi-year dataset saved to reduced_lyon_buildings_all_years.csv.
